### Procesamiento de Lenguaje Natural I
# **Desafío 1**



### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [5]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [7]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [8]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [9]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [10]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [11]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [ ]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [12]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [13]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [14]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [15]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [16]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [17]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [18]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [19]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [20]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [21]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [22]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [23]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [24]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


---

# Resolución del Desafío 1

## Consigna 1: Vectorizar documentos

Selección de los documentos

In [25]:
idx_lst = [4533, 5891, 2156, 9751, 1469]

In [26]:
for idx in idx_lst:
  print(newsgroups_train.data[idx])
  print('Clase:', newsgroups_train.target_names[y_train[idx]])
  print('---')


<tale of bike-eating-devil-dog deleted>


Come to Louisiana where it is LEGAL to carry concealed weapons on a bike!

 ----===== DoD #8177 = Technician(Dr. Speed) .NOT. Student =====----
Clase: rec.motorcycles
---
First of all I never said the Holocaust. I said before the
Holocaust. I'm not ignorant of the Holocaust and know more
about Nazi Germany than most people (maybe including you). 
	What I resent is ignorant statements that call people
names when they disagree with your position. Opposing the
atrocities commited by the Israeli governement hardly qualifies
as anti-semitism. If you think name calling is a valid form of
argument in intellectual circles, you need to get out more
often.
	I don't think the suffering of some Jews during WWII
justifies the crimes commited by the Israeli government. Any
attempt to call Civil liberterians like myself anti-semetic is
not appreciated.

Clase: talk.politics.mideast
---

Please tell me what you think would have happened had the people 
come o

**Cálculo de similaridad del coseno**

In [27]:
cossim_lst = []
for idx in idx_lst:
  cossim = cosine_similarity(X_train[idx], X_train)[0]
  cossim_lst.append(cossim)

**Obtención de los 5 documentos más similares de cada uno**

In [28]:
mostsim_lst = []
for cossim in cossim_lst:
  mostsim = np.argsort(cossim)[::-1][1:6]
  mostsim_lst.append(mostsim)
  print(mostsim)


[9737 8426   94 8134   47]
[9640 5027 3754 8224 5336]
[9284 6791 7349  649 6894]
[ 9670 10836  5200  9623   913]
[8947  561 3878 9203 4531]


Inspeccionamos la clase del documento original y la de los documentos más similares

In [29]:
doc = 0

for mostsim in mostsim_lst:
  print(f'Documento {doc+1}:')
  print(f'El documento pertenece a la clase: {newsgroups_train.target_names[y_train[idx_lst[doc]]]}')
  print('Los documentos más similares son de las clases:')
  for i in mostsim:
    print(newsgroups_train.target_names[y_train[i]])
  print('---')
  doc += 1

Documento 1:
El documento pertenece a la clase: rec.motorcycles
Los documentos más similares son de las clases:
rec.motorcycles
rec.motorcycles
rec.motorcycles
rec.motorcycles
rec.motorcycles
---
Documento 2:
El documento pertenece a la clase: talk.politics.mideast
Los documentos más similares son de las clases:
talk.politics.mideast
talk.politics.mideast
talk.politics.mideast
talk.politics.mideast
talk.politics.mideast
---
Documento 3:
El documento pertenece a la clase: talk.politics.guns
Los documentos más similares son de las clases:
talk.politics.guns
talk.politics.guns
talk.politics.guns
talk.religion.misc
talk.politics.guns
---
Documento 4:
El documento pertenece a la clase: soc.religion.christian
Los documentos más similares son de las clases:
soc.religion.christian
alt.atheism
alt.atheism
talk.politics.mideast
alt.atheism
---
Documento 5:
El documento pertenece a la clase: sci.crypt
Los documentos más similares son de las clases:
rec.sport.hockey
comp.windows.x
comp.sys.mac.har

Del resultado de la celda anterior, puede verse que los documentos similares a los documentos 1, 2 y 3 pertenecen a la misma clase que el documento original (solo una diferencia en la cuarta posición del documento 3).

En el documento 4 hay mayor variedad, pero aún así el documento más similar pertenece a la misma clase y 3 de los restantes a una clase que está en un campo semántico similar (religión).

En el documento 5 hay resultados mucho más variados. Veamos los documentos más similares al documento 5:

In [30]:
# Visualización del documento 5
print(f'Documento 5:')
print(newsgroups_train.data[idx_lst[4]])

Documento 5:

Other alternatives include output of vmstat, iostat, pstat and friends
with various flags, or even better crash. 

e.g. on an RS/6000 (AIX 3.2) you can get lots of relatively
unpredicatble data out of crash. (the output from the following script 
usually gives about 600k of goo on a moderately busy system.)

#!/bin/sh
crash <<!
proc -
tty
stack
pcb
callout
vfs -
socket
vnode
inode -
mbuf
file
mst
buffer
le
!



In [31]:
# Visualización de los 5 documentos más similares al documento 5
sim_doc = 0
print('Los 5 documentos más similares al documento 5:')
for i in mostsim_lst[4]:
  sim_doc += 1
  print(f'Documento {sim_doc}:')
  print(newsgroups_train.data[i])
  print('---')


Los 5 documentos más similares al documento 5:
Documento 1:
How do you beat the Penguins?


Crash the team plane.

---
Documento 2:

Would if only it were true ...

If only MIT would fix the !@&$^*@ twm "InstallWindowColormaps()" crash bug
once and for all, then I could say that I've (almost) unable to crash either
"twm" or "tvtwm", which would be a remarkable feat - and most desirable to
boot.  I mean, this bug has only been reported, oh, a zillion times by now ...

Now *servers*, on the other hand ... (want to crash an OpenWindows 3.0 "xnews"
server at will?  Just do an 'xbiff -xrm "XBiff*shapeWindow: on"'.  Blammo.)

---
Documento 3:
Does anyone work with the A/ROSE card?

We have the problem that after certain crashes the card disappears from the
system, and lets crash the Mac then.

Okay, we don't use the card quite like one should, because we simulate
errors in the 68000. Before every instruction some specified registers are
masked, eg. to simulate a stuck-at-1-error in certain b

**Conclusión sobre el documento 5**

Puede verse que el documento 5 tiene una longitud corta y un vocabulario limitado, lo que puede llevar a que la similaridad se base en términos comunes que no son representativos del tema específico del documento. Esto puede resultar en una clasificación menos precisa.

Al inspeccionar los 5 documentos más similares, puede verse, por ejemplo, que el documento más similar también es corto y que incluye la palabra `crash`, que el documento original la menciona 3 veces. Esta palabra también se encuentra en los documentos 2, 3 y 5.

## Consigna 2: Modelo de clasificación por prototipos (tipo zero-shot)

In [ ]:
## Consigna 2: Modelo de clasificación por prototipos (tipo zero-shot)

# Clasificar los documentos de un conjunto de test comparando cada uno 
# con todos los de entrenamiento y asignar la clase al label del documento 
# del conjunto de entrenamiento con mayor similaridad.
j = 0
y_pred_prototipo = []
for i in range(X_test.shape[0]):
  print(f'Clasificando documento {j+1} de {X_test.shape[0]}')
  cossim = cosine_similarity(X_test[i], X_train)[0]
  mostsim = np.argsort(cossim)[::-1][0]
  y_pred_prototipo.append(y_train[mostsim])
  j += 1

f1_score(y_test, y_pred_prototipo, average='macro')

Clasificando documento 1 de 7532
Clasificando documento 2 de 7532
Clasificando documento 3 de 7532
Clasificando documento 4 de 7532
Clasificando documento 5 de 7532
Clasificando documento 6 de 7532
Clasificando documento 7 de 7532
Clasificando documento 8 de 7532
Clasificando documento 9 de 7532
Clasificando documento 10 de 7532
Clasificando documento 11 de 7532
Clasificando documento 12 de 7532
Clasificando documento 13 de 7532
Clasificando documento 14 de 7532
Clasificando documento 15 de 7532
Clasificando documento 16 de 7532
Clasificando documento 17 de 7532
Clasificando documento 18 de 7532
Clasificando documento 19 de 7532
Clasificando documento 20 de 7532
Clasificando documento 21 de 7532
Clasificando documento 22 de 7532
Clasificando documento 23 de 7532
Clasificando documento 24 de 7532
Clasificando documento 25 de 7532
Clasificando documento 26 de 7532
Clasificando documento 27 de 7532
Clasificando documento 28 de 7532
Clasificando documento 29 de 7532
Clasificando documento 

0.5036472862947965

Aplicando el modelo de clasificación por prototipos, se obtuvo un F1-score de 0.5. Además, se observa que la ejecución resulta bastante más lenta que la del modelo Naive Bayes Multinomial.

## Consigna 3: Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación

De la consigna del desafío, sabemos que el F1-score macro obtenido para un MultinomialNB con parámetros por defecto (tanto del vectorizador como del modelo) es de 0.585.

### Configuración 1 del vectorizador

Entrenaremos 4 modelos con una configuración del vectorizador para evaluar el rendimiento de distintos modelos. En esta configuración, utilizaremos `stop_words='english'` para remover este tipo de palabras. Además, configuramos `max_features=5000`, que sirve para limitar el vocabulario a la cantidad de palabras especificada, ordenadas de mayor a menor por su frecuencia. En etapa de pruebas se utilizaron distintos valores de `min_df` y `max_df`, pero no generaron cambios significativos.

In [ ]:

# Configuración 1 del vectorizador
tfidfvect_1 = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_1 = tfidfvect_1.fit_transform(newsgroups_train.data)
X_test_1 = tfidfvect_1.transform(newsgroups_test.data)

# Modelo 1: MultinomialNB por defecto
clf_1_1 = MultinomialNB()
clf_1_1.fit(X_train_1, y_train)
y_pred_1_1 = clf_1_1.predict(X_test_1)
f1_score(y_test, y_pred_1_1, average='macro')
print(f'F1-score Modelo 1: {f1_score(y_test, y_pred_1_1, average="macro")}')

# Modelo 2: ComplementNB por defecto
clf_1_2 = ComplementNB()
clf_1_2.fit(X_train_1, y_train)
y_pred_1_2 = clf_1_2.predict(X_test_1)
f1_score(y_test, y_pred_1_2, average='macro')
print(f'F1-score Modelo 2: {f1_score(y_test, y_pred_1_2, average="macro")}')

# Modelo 3: MultinomialNB con alpha=0.5
clf_1_3 = MultinomialNB(alpha=0.5)
clf_1_3.fit(X_train_1, y_train)
y_pred_1_3 = clf_1_3.predict(X_test_1)
f1_score(y_test, y_pred_1_3, average='macro')
print(f'F1-score Modelo 3: {f1_score(y_test, y_pred_1_3, average="macro")}')

# Modelo 4: ComplementNB con alpha=0.5
clf_1_4 = ComplementNB(alpha=0.5)
clf_1_4.fit(X_train_1, y_train)
y_pred_1_4 = clf_1_4.predict(X_test_1)
f1_score(y_test, y_pred_1_4, average='macro')
print(f'F1-score Modelo 4: {f1_score(y_test, y_pred_1_4, average="macro")}')



F1-score Modelo 1: 0.6255561324700291
F1-score Modelo 2: 0.6360789993932491
F1-score Modelo 3: 0.6334945664704279
F1-score Modelo 4: 0.6340505375807955


### Configuración 2 del vectorizador

En este caso, utilizamos los mismos parámetros, pero modificamos el parámetro `max_features` a 40.000. Vemos que este cambio mejora los resultados. Se probó aumentar más este valor, sin obtener beneficios.

In [ ]:
# Configuración 2 del vectorizador
tfidfvect_2 = TfidfVectorizer(stop_words='english', max_features=40000)
X_train_2 = tfidfvect_2.fit_transform(newsgroups_train.data)
X_test_2 = tfidfvect_2.transform(newsgroups_test.data)

# Modelo 5: MultinomialNB con configuración 2 del vectorizador
clf_2_1 = MultinomialNB()
clf_2_1.fit(X_train_2, y_train)
y_pred_2_1 = clf_2_1.predict(X_test_2)
f1_score(y_test, y_pred_2_1, average='macro')
print(f'F1-score Modelo 5: {f1_score(y_test, y_pred_2_1, average="macro")}')

# Modelo 6: ComplementNB con configuración 2 del vectorizador
clf_2_2 = ComplementNB()
clf_2_2.fit(X_train_2, y_train)
y_pred_2_2 = clf_2_2.predict(X_test_2)
f1_score(y_test, y_pred_2_2, average='macro')
print(f'F1-score Modelo 6: {f1_score(y_test, y_pred_2_2, average="macro")}')

# Modelo 7: MultinomialNB con alpha=0.5 y configuración 2 del vectorizador
clf_2_3 = MultinomialNB(alpha=0.5)
clf_2_3.fit(X_train_2, y_train)
y_pred_2_3 = clf_2_3.predict(X_test_2)
f1_score(y_test, y_pred_2_3, average='macro')
print(f'F1-score Modelo 7: {f1_score(y_test, y_pred_2_3, average="macro")}')

# Modelo 8: ComplementNB con alpha=0.5 y configuración 2 del vectorizador
clf_2_4 = ComplementNB(alpha=0.5)
clf_2_4.fit(X_train_2, y_train)
y_pred_2_4 = clf_2_4.predict(X_test_2)
f1_score(y_test, y_pred_2_4, average='macro')
print(f'F1-score Modelo 8: {f1_score(y_test, y_pred_2_4, average="macro")}')

F1-score Modelo 5: 0.6507092781495901
F1-score Modelo 6: 0.6948086067114924
F1-score Modelo 7: 0.6632121056852869
F1-score Modelo 8: 0.6965270145599776


### Conclusiones consigna 3

De los 8 modelos entrenados (cuatro con una configuración del vectorizador y otros cuatro con otra configuración), puede verse que vectorizar con los parámetros `max_features` y `stop_words` mejora el rendimiento del modelo. En la primera configuración, se utilizó `max_features=5000` y en la segunda `max_features=40000`, observando mejores resultados con la 2da. También se observó que seguir aumentando este valor no mejoraba el desempeño. Vale recordar que el tamaño total del vocabulario es de aproximadamente 101 mil palabras, es decir que el mejor rendimiento se obtuvo con el 40% de la cantidad de palabras. Por otra parte, como se comentó anteriormente, Los parámetros `max_df` y `min_df` no generaron cambios significativos.

Respecto al modelo, se obtuvieron mejores resultados con el modelo `ComplementNB` sobre el `MultinomialNB`. Utilizar un valor de alpha para la inicialización del modelo no mostró mejores resultados. El mejor valor de F1-score obtenido (modelo 8) fue de 0.695, sobre el 0.585 obtenido con las configuraciones por defecto.

## Consigna 4

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


In [43]:
## Consigna 4: transponer la matriz documento-termino

# Transpuesta de la matriz documento-término
X_train_T = X_train.T

print(f"Dimensión de la matriz documento-término: {X_train.shape}")
print(f"Dimensión de la matriz término-documento: {X_train_T.shape}")

words = ['moon', 'voltage', 'fuel', 'computer', 'jesus']
words_idx = [tfidfvect.vocabulary_[p] for p in words]

for palabra, idx_palabra in zip(words, words_idx):
    wordsim = cosine_similarity(X_train_T[idx_palabra], X_train_T)[0]

    mostsimwords = np.argsort(wordsim)[::-1][1:6]

    print(f"\nLas 5 palabras más similares a '{palabra}' son:")
    for i in mostsimwords:
        print(f"  {idx2word[i]:15s}")

Dimensión de la matriz documento-término: (11314, 101631)
Dimensión de la matriz término-documento: (101631, 11314)

Las 5 palabras más similares a 'moon' son:
  lunar          
  phases         
  gravitacional  
  atraction      
  sattellite     

Las 5 palabras más similares a 'voltage' son:
  1v             
  10vdc          
  vdcmax         
  60vdc          
  crts           

Las 5 palabras más similares a 'fuel' son:
  gouged         
  perturbations  
  gallon         
  injector       
  gauge          

Las 5 palabras más similares a 'computer' son:
  decwriter      
  deluged        
  harkens        
  shopper        
  the            

Las 5 palabras más similares a 'jesus' son:
  christ         
  god            
  kingdom        
  mat            
  bible          


La matriz término-documento generada permite obtener una representación vectorial de las palabras basada en el documento al que pertenecen. Así, palabras que pertenecen consistentemente a los mismos documentos tendrán mayor similitud.

De acuerdo a los resultados obtenidos, podemos observar que para las palabras con una clara pertenencia a un campo semántico (jesus, voltage, moon), se obtienen palabras que están fuertemente vinculadas. Sin embargo, para palabras más generales como computer, se obtienen palabras que no están tan directamente vinculadas.

Por lo tanto, este método puede resultar útil para casos puntuales, pero presenta fuertes limitaciones.